# Refactor KPConv (v2)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import math
import random
import warnings
from pathlib import Path
from typing import Any, Literal, Tuple, Union

import torch
import torch.nn as nn
from torch import Tensor
from torch.nn.init import kaiming_uniform_
from torch.nn.parameter import Parameter
import torch_scatter
from torch_cluster import knn, knn_graph

from torch_pointcloud.utils.config import CACHE_DIR
from torch_pointcloud.utils.geometry import rodrigues_rotation_matrix, spherical_points_gradient, spherical_points_lloyd

from torch_pointcloud.models.kpconv import KPConv

In [3]:
import sys
sys.path.append("/home/arthur/Documents/Code/Github/Easy-KPConv")

### KPConv

In [4]:
torch.manual_seed(42)
kpconv = KPConv(
    spatial_dim=3,
    in_channels=3,
    out_channels=64,
    kernel_size=15,
    kp_sigma=0.5,
    kp_radius=0.5,
    deformable=False,
    aggregation_mode="sum",
)
kpconv

KPConv(in_channels=3, out_channels=64, kp_radius=0.5, kp_sigma=0.5, kp_influence='linear', fixed_kernel_points='center', aggregation_mode='sum', )

In [5]:
from easy_kpconv.layers.kpconv import KPConv as EasyKPConv
from easy_kpconv.layers.kpconv_blocks import KPResidualBlock as EasyKPResidualBlock

torch.manual_seed(42)
easy_kpconv = EasyKPConv(
    in_channels=3,
    out_channels=64,
    kernel_size=15,
    radius=0.5,
    sigma=0.5,
    groups=1,
    bias=False,
    dimension=3,
    inf=1e6,
)

easy_kpconv.kernel_points = kpconv.kernel.clone()
# easy_kpconv.weights = kpconv.weights.clone()
# easy_kpconv.bias = kpconv.bias.clone()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [6]:
torch.manual_seed(42)
coords = torch.randn(100, 3)
coords_s = torch.randn(100, 3)
features = torch.randn(100, 3)
batch = torch.cat([torch.zeros(50), torch.ones(50)]).long()
# edge_index = knn_graph(coords, 15, batch=batch, loop=False, flow="source_to_target")
# neighbor_idxs = edge_index[0].reshape(-1, 15)

edge_index = knn(coords, coords_s, 15, batch_x=batch, batch_y=batch)
neighbor_idxs = edge_index[1].reshape(-1, 15)
neighbor_idxs

tensor([[ 5, 44, 21,  ..., 28, 35, 29],
        [49, 25,  9,  ...,  8, 40, 12],
        [17, 47, 23,  ..., 11, 37, 35],
        ...,
        [79, 54, 76,  ..., 59, 81, 89],
        [91, 53, 57,  ..., 54, 64, 92],
        [50, 63, 76,  ..., 56, 82, 89]])

In [7]:
print(edge_index[0].max(), edge_index[1].max())
print(f"{coords.shape = }")
print(f"{features.shape = }")

kpconv(features.cpu(), coords.cpu(), coords.cpu(), edge_index.cpu())

tensor(99) tensor(99)
coords.shape = torch.Size([100, 3])
features.shape = torch.Size([100, 3])


tensor([[ 0.0035, -0.1898,  0.1773,  ...,  0.0887, -0.0212, -0.1980],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0011,  0.0035,  0.0066,  ..., -0.0004, -0.0022, -0.0047],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0021,  0.0157, -0.0922,  ..., -0.2240, -0.1103, -0.1677],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       grad_fn=<ScatterAddBackward0>)

In [8]:
easy_kpconv(coords.cpu(), coords.cpu(), features.cpu(), neighbor_idxs.cpu()) # .reshape(-1, 3) # .reshape(-1, 15)

tensor([[ 0.0035, -0.1898,  0.1773,  ...,  0.0887, -0.0212, -0.1980],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0011,  0.0035,  0.0066,  ..., -0.0004, -0.0022, -0.0047],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0021,  0.0157, -0.0922,  ..., -0.2240, -0.1103, -0.1677],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       grad_fn=<ViewBackward0>)

In [9]:
# from torch_geometric.nn.pool import radius_graph, radius

# torch.manual_seed(42)
# # q_points = torch.arange(50 * 3, dtype=torch.float, device="cuda").reshape(50, 3)
# # s_points = torch.arange(50 * 3, dtype=torch.float, device="cuda").reshape(50, 3) + 2
# q_points = torch.randn(1000, 3, device="cuda")
# s_points = torch.randn(1000, 3, device="cuda")

# q_lengths = torch.tensor([500, 500], device="cuda")
# s_lengths = torch.tensor([500, 500], device="cuda")

# q_batch = torch.cat([torch.zeros(500), torch.ones(500)]).long().to("cuda")
# s_batch = torch.cat([torch.zeros(500), torch.ones(500)]).long().to("cuda")

# edge_index = radius(q_points, s_points, r=2, batch_x=q_batch, batch_y=s_batch, max_num_neighbors=5)
# neighbor_idxs = edge_index[1].reshape(-1, 5)
# neighbor_idxs

In [10]:
# kpconv(features.cpu(), q_points.cpu(), s_points.cpu(), edge_index.cpu())

In [11]:
# easy_kpconv(q_points.cpu(), s_points.cpu(), features.cpu(), neighbor_idxs.cpu())

In [12]:
import sys
sys.path.append("/home/arthur/Documents/Code/Github/Easy-KPConv/examples/scene_segmentation")

from easy_kpconv.ops.graph_pyramid import build_grid_and_radius_graph_pyramid
from dataset import train_valid_data_loader
from config import make_cfg
from model import create_model


cfg = make_cfg()
easy_model = create_model(cfg).cuda()
train_loader, val_loader = train_valid_data_loader(cfg, "Area_1")
data_dict = next(iter(train_loader))

feats = data_dict["feats"].cuda()
points = data_dict["points"].cuda()
lengths = data_dict["lengths"].cuda()

graph_pyramid = build_grid_and_radius_graph_pyramid(
    points, 
    lengths, 
    num_stages=2, 
    voxel_size=easy_model.voxel_size, 
    search_radius=0.5, 
    neighbor_limits=[5, 5]
)

points_list = graph_pyramid["points"]
neighbors_list = graph_pyramid["neighbors"]
subsampling_list = graph_pyramid["subsampling"]
upsampling_list = graph_pyramid["upsampling"]
lengths = graph_pyramid["lengths"]

Testing data is cached in '/home/arthur/Documents/Code/Github/Easy-KPConv/examples/scene_segmentation/dataset_s3dis/s3dis_voxelized_testing_cached_2.5b_1s_2048t'.
i = 0 | voxel_size = 0.04 | points.shape = torch.Size([176321, 3])
i = 1 | voxel_size = 0.08 | points.shape = torch.Size([55263, 3])
--------------------------------
i = 0
sub_points.shape = torch.Size([55263, 3])
cur_points.shape = torch.Size([176321, 3])
sub_lengths.shape = torch.Size([16])
cur_lengths.shape = torch.Size([16])
search_radius = 0.5
neighbor_limits[i] = 5
sub_points = tensor([[-1.2021,  0.3889,  2.2597],
        [-1.2069,  0.3987,  2.4875],
        [-1.2172,  0.4607,  0.7058],
        ...,
        [ 1.1681,  0.2555,  2.5781],
        [ 1.2034,  0.2549,  0.9296],
        [ 1.2017,  0.2654,  1.7216]], device='cuda:0')
cur_points = tensor([[-0.2896,  1.6124,  0.6178],
        [-0.2623,  1.5890,  3.2165],
        [-0.2661,  1.5903,  0.6264],
        ...,
        [ 0.1841, -1.3739,  0.6979],
        [ 0.1796, -1.39

In [13]:
easy_model.cuda()
data_dict = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in data_dict.items()}
easy_model(data_dict)

feats.shape=torch.Size([176321, 4])
points.shape=torch.Size([176321, 3])
lengths.shape=torch.Size([16])
i = 0 | voxel_size = 0.04 | points.shape = torch.Size([176321, 3])
i = 1 | voxel_size = 0.08 | points.shape = torch.Size([55263, 3])
i = 2 | voxel_size = 0.16 | points.shape = torch.Size([14945, 3])
i = 3 | voxel_size = 0.32 | points.shape = torch.Size([4019, 3])
i = 4 | voxel_size = 0.64 | points.shape = torch.Size([1144, 3])
--------------------------------
i = 0
sub_points.shape = torch.Size([55263, 3])
cur_points.shape = torch.Size([176321, 3])
sub_lengths.shape = torch.Size([16])
cur_lengths.shape = torch.Size([16])
search_radius = 0.1
neighbor_limits[i] = 24
sub_points = tensor([[-1.2021,  0.3889,  2.2597],
        [-1.2069,  0.3987,  2.4875],
        [-1.2172,  0.4607,  0.7058],
        ...,
        [ 1.1681,  0.2555,  2.5781],
        [ 1.2034,  0.2549,  0.9296],
        [ 1.2017,  0.2654,  1.7216]], device='cuda:0')
cur_points = tensor([[-0.2896,  1.6124,  0.6178],
        [

{'scores': tensor([[-0.2693, -0.1212,  0.5078,  ...,  0.0884,  0.0069,  0.5156],
         [-0.1406, -0.1664,  0.3448,  ...,  0.4771,  0.0416,  0.5080],
         [-0.2801, -0.1629,  0.3656,  ...,  0.1389,  0.0062,  0.4407],
         ...,
         [-0.1449, -0.0883,  0.5768,  ...,  0.0588,  0.1467,  0.5531],
         [-0.2361,  0.0499,  0.4376,  ...,  0.2389, -0.1214,  0.4167],
         [ 0.1101,  0.0209,  0.3565,  ...,  0.1936,  0.1356,  0.3994]],
        device='cuda:0', grad_fn=<AddmmBackward0>)}

In [14]:
from torch_cluster import radius
from torch_pointcloud.utils.conversion import bincount_to_batch

radius(
    points_list[0].cpu(), 
    points_list[1].cpu(), 
    r=0.5, 
    batch_x=bincount_to_batch(lengths[0].cpu()), 
    batch_y=bincount_to_batch(lengths[1].cpu()),
    max_num_neighbors=5,
).shape, subsampling_list[0].flatten().shape


(torch.Size([2, 276315]), torch.Size([276315]))

In [15]:
from torch_geometric.nn.pool import radius
from torch_pointcloud.utils.conversion import bincount_to_batch
torch.manual_seed(42)
feats = torch.randn(len(points_list[0]), 3)
edge_index = torch.row_stack([
    torch.arange(len(subsampling_list[0])).unsqueeze(-1).repeat(1, 5).flatten().cpu(),
    subsampling_list[0].flatten().cpu(),
])
# edge_index = radius(
#     points_list[0].cpu(), 
#     points_list[1].cpu(), 
#     r=50, 
#     batch_x=bincount_to_batch(lengths[0].cpu()), 
#     batch_y=bincount_to_batch(lengths[1].cpu()),
#     max_num_neighbors=5,
# )
print(edge_index[0].max(), edge_index[1].max())
print(f"{points_list[1].shape = }")
print(f"{points_list[0].shape = }")
print(f"{feats.shape = }")
print(f"{subsampling_list[0].shape = }")

kpconv(feats.cpu(), points_list[1].cpu(), points_list[0].cpu(), edge_index.cpu())

tensor(55262) tensor(176320)
points_list[1].shape = torch.Size([55263, 3])
points_list[0].shape = torch.Size([176321, 3])
feats.shape = torch.Size([176321, 3])
subsampling_list[0].shape = torch.Size([55263, 5])


tensor([[ 2.1396e-02,  1.1502e-01, -2.5107e-01,  ..., -9.9126e-02,
          2.3475e-02, -7.2690e-02],
        [-1.4040e-01, -4.6512e-02,  3.6345e-04,  ...,  8.7076e-02,
          8.6443e-02,  2.0887e-01],
        [ 1.6437e-01,  7.3075e-02,  3.6367e-01,  ...,  3.1553e-02,
         -1.2431e-01, -2.2247e-01],
        ...,
        [ 7.2786e-02,  1.1072e-01,  2.0343e-01,  ...,  3.2520e-03,
         -7.9134e-02,  4.0577e-02],
        [-2.7808e-01, -4.8468e-01, -1.4146e-01,  ...,  1.6833e-01,
          3.5287e-02,  2.3355e-01],
        [-5.5587e-02, -3.6187e-01,  4.0370e-01,  ...,  2.9247e-01,
         -4.9406e-02, -1.0345e-01]], grad_fn=<ScatterAddBackward0>)

In [16]:
torch.manual_seed(42)
feats = torch.randn(len(points_list[0]), 3)
easy_kpconv(points_list[1].cpu(), points_list[0].cpu(), feats.cpu(), subsampling_list[0].cpu()) # .permute(0, 2, 1).reshape(-1, 15)

tensor([[ 2.1396e-02,  1.1502e-01, -2.5107e-01,  ..., -9.9126e-02,
          2.3475e-02, -7.2690e-02],
        [-1.4040e-01, -4.6512e-02,  3.6341e-04,  ...,  8.7076e-02,
          8.6443e-02,  2.0887e-01],
        [ 1.6437e-01,  7.3075e-02,  3.6367e-01,  ...,  3.1553e-02,
         -1.2431e-01, -2.2247e-01],
        ...,
        [ 7.2786e-02,  1.1072e-01,  2.0343e-01,  ...,  3.2520e-03,
         -7.9134e-02,  4.0577e-02],
        [-2.7808e-01, -4.8468e-01, -1.4146e-01,  ...,  1.6833e-01,
          3.5287e-02,  2.3355e-01],
        [-5.5587e-02, -3.6187e-01,  4.0370e-01,  ...,  2.9247e-01,
         -4.9407e-02, -1.0345e-01]], grad_fn=<ViewBackward0>)

### KPResidualBlock

In [17]:
from torch_pointcloud.models.kpconv import KPResidualBlock
from easy_kpconv.layers.kpconv_blocks import KPResidualBlock as EasyKPResidualBlock

In [18]:
torch.manual_seed(42)
block = KPResidualBlock(
    spatial_dim=3,
    in_channels=3,
    out_channels=64,
    kernel_size=15,
    kp_radius=0.5,
    kp_sigma=0.5,
    strided=False,
    norm=None,
    act=None,
    # norm="layer_norm",
    # act="leaky_relu",
)

In [19]:
torch.manual_seed(42)
easy_block = EasyKPResidualBlock(
    in_channels=3,
    out_channels=64,
    kernel_size=15,
    radius=0.5,
    sigma=0.5,
    groups=1,
    dimension=3,
    strided=False,
    # norm_cfg="LayerNorm",
    # act_cfg="LeakyReLU",
    norm_cfg=None,
    act_cfg=None,
)

In [20]:
easy_block.conv.conv.kernel_points = block.conv.conv.kernel.clone()
easy_block.unary2.mlp.bias = block.unary2.mlp.bias
easy_block.unary2.mlp.weight = block.unary2.mlp.weight
easy_block.unary_shortcut.mlp.weight = block.shortcut.mlp.weight
easy_block.unary_shortcut.mlp.bias = block.shortcut.mlp.bias

In [21]:
torch.manual_seed(42)
feats = torch.randn(len(points_list[0]), 3)

edge_index = torch.row_stack([
    torch.arange(len(neighbors_list[0])).unsqueeze(-1).repeat(1, 5).flatten().cpu(),
    neighbors_list[0].flatten().cpu(),
])

In [22]:
block(feats.cpu(), points_list[0].cpu(), points_list[0].cpu(), edge_index.cpu())

tensor([[-1.2483,  0.3174, -0.1025,  ...,  0.7529,  0.7387, -1.4006],
        [ 1.3254,  0.2059,  1.6340,  ..., -0.4628, -0.7481,  0.1666],
        [ 1.4654,  0.5776,  0.7721,  ...,  1.2032,  0.9192,  1.1067],
        ...,
        [ 0.8318,  0.2336,  0.2071,  ...,  0.4190,  0.1388,  0.0196],
        [ 0.8937,  0.0946,  1.1629,  ..., -0.3186, -0.6936, -0.1165],
        [ 1.6047,  0.2389,  1.8213,  ..., -0.4499, -0.9331,  0.4478]],
       grad_fn=<AddBackward0>)

In [23]:
easy_block(points_list[0].cpu(), points_list[0].cpu(), feats.cpu(), neighbors_list[0].cpu())

tensor([[-1.2483,  0.3174, -0.1025,  ...,  0.7529,  0.7387, -1.4006],
        [ 1.3254,  0.2059,  1.6340,  ..., -0.4628, -0.7481,  0.1666],
        [ 1.4654,  0.5776,  0.7721,  ...,  1.2032,  0.9192,  1.1067],
        ...,
        [ 0.8318,  0.2336,  0.2071,  ...,  0.4190,  0.1388,  0.0196],
        [ 0.8937,  0.0946,  1.1629,  ..., -0.3186, -0.6936, -0.1165],
        [ 1.6047,  0.2389,  1.8213,  ..., -0.4499, -0.9331,  0.4478]],
       grad_fn=<AddBackward0>)

In [24]:
torch.manual_seed(42)

feats = torch.randn(len(points_list[0]), 3)
edge_index = torch.row_stack([
    torch.arange(len(subsampling_list[0])).unsqueeze(-1).repeat(1, 5).flatten().cpu(),
    subsampling_list[0].flatten().cpu(),
])

print(f"{points_list[1].shape = }, {points_list[0].shape = }, {feats.shape = }, {edge_index.shape = }")

points_list[1].shape = torch.Size([55263, 3]), points_list[0].shape = torch.Size([176321, 3]), feats.shape = torch.Size([176321, 3]), edge_index.shape = torch.Size([2, 276315])


In [25]:
block.strided = True
block(feats.cpu(), points_list[1].cpu(), points_list[0].cpu(), edge_index.cpu())

tensor([[-1.0632,  0.0889, -0.3216,  ...,  0.1375, -0.0259, -1.7412],
        [-0.7276,  0.0246,  0.1070,  ...,  0.1467,  0.0027, -1.3017],
        [-1.4100,  0.2848,  0.0860,  ...,  0.2663,  0.4211, -1.7174],
        ...,
        [-0.8848,  0.0757,  0.1303,  ...,  0.0840,  0.0885, -1.3311],
        [ 0.0431,  0.0348,  0.3574,  ...,  0.1712, -0.3496, -0.6630],
        [-0.8534,  0.0489, -0.3804,  ...,  0.3374,  0.0278, -1.2839]],
       grad_fn=<AddBackward0>)

In [26]:
easy_block.strided = True
easy_block(points_list[1].cpu(), points_list[0].cpu(), feats.cpu(), subsampling_list[0].cpu())

tensor([[-1.0632,  0.0889, -0.3216,  ...,  0.1375, -0.0259, -1.7412],
        [-0.7276,  0.0246,  0.1070,  ...,  0.1467,  0.0027, -1.3017],
        [-1.4100,  0.2848,  0.0860,  ...,  0.2663,  0.4211, -1.7174],
        ...,
        [-0.8848,  0.0757,  0.1303,  ...,  0.0840,  0.0885, -1.3311],
        [ 0.0431,  0.0348,  0.3574,  ...,  0.1712, -0.3496, -0.6630],
        [-0.8534,  0.0489, -0.3804,  ...,  0.3374,  0.0278, -1.2839]],
       grad_fn=<AddBackward0>)

In [27]:
# points_list[1].shape = torch.Size([55263, 3]), 
# points_list[0].shape = torch.Size([176321, 3]), 
# feats.shape = torch.Size([176321, 3]), 
# edge_index.shape = torch.Size([2, 276315])

In [28]:
radius(
    points_list[0].cpu(), 
    points_list[1].cpu(), 
    r=0.5, 
    batch_x=bincount_to_batch(lengths[0].cpu()), 
    batch_y=bincount_to_batch(lengths[1]).cpu(),
    max_num_neighbors=5,
)

tensor([[     0,      0,      0,  ...,  55262,  55262,  55262],
        [ 11273,  11212,  11211,  ..., 165972, 166009, 165973]])

In [29]:
from torch_pointcloud.models.kpconv import EncoderBlock, GridPool


encoder_block = EncoderBlock(
    spatial_dim=3,
    in_channels=3,
    out_channels=64,
    depth=6,
    kernel_size=15,
    kp_radius=0.5,
    kp_sigma=0.5,
    radius=0.5,
    max_num_neighbors=5,
    norm=None,
    act=None,
    downsample=GridPool(grid_size=0.08, reduce="max"),
)

encoder_block

EncoderBlock(
  (downsample): GridPool(grid_size=0.08, reduce='max')
  (blocks): ModuleList(
    (0): KPResidualBlock(
      (unary1): UnaryBlock(
        (mlp): Linear(in_features=3, out_features=16, bias=True)
      )
      (conv): KPConvBlock(
        (conv): KPConv(in_channels=16, out_channels=16, kp_radius=0.5, kp_sigma=0.5, kp_influence='linear', fixed_kernel_points='center', aggregation_mode='sum', )
      )
      (unary2): UnaryBlock(
        (mlp): Linear(in_features=16, out_features=64, bias=True)
      )
      (shortcut): UnaryBlock(
        (mlp): Linear(in_features=3, out_features=64, bias=True)
      )
    )
    (1-5): 5 x KPResidualBlock(
      (unary1): UnaryBlock(
        (mlp): Linear(in_features=64, out_features=16, bias=True)
      )
      (conv): KPConvBlock(
        (conv): KPConv(in_channels=16, out_channels=16, kp_radius=0.5, kp_sigma=0.5, kp_influence='linear', fixed_kernel_points='center', aggregation_mode='sum', )
      )
      (unary2): UnaryBlock(
        (

In [30]:
points_list[0].shape, points_list[1].shape

(torch.Size([176321, 3]), torch.Size([55263, 3]))

In [31]:
torch.manual_seed(42)
feats = torch.randn(len(points_list[1]), 3)
encoder_block.cpu()
encoder_block(feats.cpu(), points_list[1].cpu(), bincount_to_batch(lengths[1]).cpu())

(tensor([[ 0.0372, -0.0919,  0.9790,  ..., -0.5635,  0.5639, -0.4384],
         [ 0.2531,  0.0123,  0.9491,  ..., -0.6509,  0.6341, -0.3490],
         [ 0.7450,  0.5152,  1.4227,  ..., -1.0874,  1.6053,  0.0209],
         ...,
         [ 0.9406,  0.3346,  1.9854,  ..., -1.4866,  2.0516, -0.1282],
         [ 0.5306,  0.9356,  1.5845,  ..., -1.4503,  1.3068, -0.8782],
         [ 0.9579,  0.3309,  1.9481,  ..., -1.5092,  2.0852, -0.1526]],
        grad_fn=<AddBackward0>),
 tensor([[ 0.7262, -1.1364,  0.6107],
         [ 0.6487, -1.0858,  0.6167],
         [ 0.7549, -1.0747,  0.6171],
         ...,
         [-0.1212,  0.9117,  2.6543],
         [-0.0230,  0.9342,  2.6433],
         [-0.0844,  0.9707,  2.6453]]),
 tensor([ 0,  0,  0,  ..., 15, 15, 15]))

In [32]:
torch.manual_seed(42)
feats = torch.randn(len(points_list[0]), 3)
encoder_block.cuda()
encoder_block(feats.cuda(), points_list[0].cuda(), bincount_to_batch(lengths[0]).cuda())

(tensor([[ 0.6796,  0.5492,  1.1398,  ..., -1.1077,  1.3100, -0.7805],
         [ 0.7025,  0.5629,  1.1489,  ..., -1.0914,  1.2681, -0.7607],
         [ 0.6551,  0.5549,  1.1855,  ..., -1.1430,  1.3001, -0.7246],
         ...,
         [ 1.1509,  1.8005,  2.0032,  ..., -1.6986,  2.0537, -0.8080],
         [ 1.3583,  1.8039,  1.9413,  ..., -1.8241,  2.0988, -0.7648],
         [ 1.3114,  1.8018,  1.9482,  ..., -1.7873,  2.0737, -0.7700]],
        device='cuda:0', grad_fn=<AddBackward0>),
 tensor([[ 0.7262, -1.1364,  0.6107],
         [ 0.6301, -1.0752,  0.6120],
         [ 0.6710, -1.1058,  0.6201],
         ...,
         [ 0.0451,  0.8777,  2.6434],
         [-0.1039,  0.9829,  2.6403],
         [-0.0681,  0.9700,  2.6448]], device='cuda:0'),
 tensor([ 0,  0,  0,  ..., 15, 15, 15], device='cuda:0'))

---

In [33]:
# feats.shape=torch.Size([176321, 4])
# points.shape=torch.Size([176321, 3])
# lengths.shape=torch.Size([16])
# i = 0 | voxel_size = 0.04 | points.shape = torch.Size([176321, 3])
# i = 1 | voxel_size = 0.08 | points.shape = torch.Size([55263, 3])
# i = 2 | voxel_size = 0.16 | points.shape = torch.Size([14945, 3])
# i = 3 | voxel_size = 0.32 | points.shape = torch.Size([4019, 3])
# i = 4 | voxel_size = 0.64 | points.shape = torch.Size([1144, 3])
# [KeOps] Generating code for KMin_ArgKMin_Reduction reduction (with parameters 0) of formula Sqrt(Sum((a-b)**2)) with a=Var(0,3,0), b=Var(1,3,1) ... OK
# --------------------------------
# i = 0
# sub_points.shape = torch.Size([55263, 3])
# cur_points.shape = torch.Size([176321, 3])
# sub_lengths.shape = torch.Size([16])
# cur_lengths.shape = torch.Size([16])
# search_radius = 0.1
# neighbor_limits[i] = 24

# [KeOps] Generating code for KMin_ArgKMin_Reduction reduction (with parameters 0) of formula Sqrt(Sum((a-b)**2)) with a=Var(0,3,0), b=Var(1,3,1) ... OK
# --------------------------------
# i = 1
# sub_points.shape = torch.Size([14945, 3])
# cur_points.shape = torch.Size([55263, 3])
# sub_lengths.shape = torch.Size([16])
# cur_lengths.shape = torch.Size([16])
# search_radius = 0.2
# neighbor_limits[i] = 40

# [KeOps] Generating code for KMin_ArgKMin_Reduction reduction (with parameters 0) of formula Sqrt(Sum((a-b)**2)) with a=Var(0,3,0), b=Var(1,3,1) ... OK
# --------------------------------
# i = 2
# sub_points.shape = torch.Size([4019, 3])
# cur_points.shape = torch.Size([14945, 3])
# sub_lengths.shape = torch.Size([16])
# cur_lengths.shape = torch.Size([16])
# search_radius = 0.4
# neighbor_limits[i] = 34

# [KeOps] Generating code for KMin_ArgKMin_Reduction reduction (with parameters 0) of formula Sqrt(Sum((a-b)**2)) with a=Var(0,3,0), b=Var(1,3,1) ... OK
# --------------------------------
# i = 3
# sub_points.shape = torch.Size([1144, 3])
# cur_points.shape = torch.Size([4019, 3])
# sub_lengths.shape = torch.Size([16])
# cur_lengths.shape = torch.Size([16])
# search_radius = 0.8
# neighbor_limits[i] = 35

In [34]:
easy_model

KPFCNN(
  (encoder1_1): KPConvBlock(
    (conv): KPConv(kernel_size=15, in_channels=5, out_channels=64, radius=0.1, sigma=0.08, bias=True, groups=1, dimension=3, inf=1e+06)
    (norm): GroupNormPackMode(8, 64, eps=1e-05, affine=True)
    (act): LeakyReLU(negative_slope=0.2)
  )
  (encoder1_2): KPResidualBlock(
    (unary1): UnaryBlockPackMode(
      (mlp): Linear(in_features=64, out_features=32, bias=True)
      (norm): GroupNormPackMode(4, 32, eps=1e-05, affine=True)
      (act): LeakyReLU(negative_slope=0.2)
    )
    (conv): KPConvBlock(
      (conv): KPConv(kernel_size=15, in_channels=32, out_channels=32, radius=0.1, sigma=0.08, bias=True, groups=1, dimension=3, inf=1e+06)
      (norm): GroupNormPackMode(4, 32, eps=1e-05, affine=True)
      (act): LeakyReLU(negative_slope=0.2)
    )
    (unary2): UnaryBlockPackMode(
      (mlp): Linear(in_features=32, out_features=128, bias=True)
      (norm): GroupNormPackMode(16, 128, eps=1e-05, affine=True)
    )
    (unary_shortcut): UnaryBlock

In [39]:
easy_model.first_radius

0.1

In [ ]:
first_radius

0.1

In [35]:
from torch_pointcloud.models.kpconv import KPConvNetClassification, create_encoder_blocks

kpconv_model = KPConvNetClassification(
    in_channels=3,
    num_classes=10,
    stem_channels=64,
    encoder_depths=[1, 3, 3, 3],
    encoder_channels=[64, 128, 256, 512],
    encoder_num_neighbors=[16, 16, 16, 16],
    grid_sizes=[0.08, 0.16, 0.32],
    kernel_size=15,
    kp_radius=[0.1, 0.2, 0.4],
    kp_sigma=0.08,
    radii=[0.1, 0.2, 0.4, 0.8],
)
kpconv_model

KPConvNetClassification(
  (stem): Sequential(
    (0): Linear(in_features=3, out_features=64, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.0, inplace=False)
  )
  (encoder): ModuleList(
    (0): EncoderBlock(
      (blocks): ModuleList(
        (0): KPResidualBlock(
          (unary1): UnaryBlock(
            (mlp): Linear(in_features=64, out_features=16, bias=True)
            (norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act): ReLU()
          )
          (conv): KPConvBlock(
            (conv): KPConv(in_channels=16, out_channels=16, kp_radius=0.1, kp_sigma=0.08, kp_influence='linear', fixed_kernel_points='center', aggregation_mode='sum', )
            (norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act): ReLU()
          )
          (unary2): UnaryBlock(
            (mlp): Linear(in_f

In [36]:
torch.manual_seed(42)
feats = torch.randn(len(points_list[0]), 3)
kpconv_model.cuda()
kpconv_model.forward(feats.cuda(), points_list[0].cuda(), bincount_to_batch(lengths[0]).cuda())

tensor([[-18.4566, -12.0439,   4.3513,  14.9024, -14.3467, -39.1720, -15.4492,
          -0.8019,  25.3412, -28.8034],
        [ -0.7217,  -1.2483,  -1.3925,  -0.1997,   0.2838,  -0.9178,   1.7491,
           1.4394,   1.5421,  -2.4672],
        [ -1.4947,  -0.3799,  -0.4800,   0.7855,   0.0534,  -1.7773,   0.2828,
          -0.2855,  -0.3879,   0.6065],
        [ -3.8861,   0.4382,  -1.0908,   2.8896,  -0.0517,  -5.5411,   2.3495,
          -0.3720,   2.0268,  -0.8121],
        [ -1.6843,  -2.0616,  -1.7451,   2.2638,   2.4230,  -6.6757,   1.2886,
           0.8664,  -0.4147,  -5.4696],
        [ -1.9779,  -4.0751,   0.5186,   2.3035,  -0.4087,  -2.4636,   1.5281,
          -1.7658,   0.5295,  -2.5497],
        [ -1.7507,  -0.8835,  -0.8731,   1.2726,  -0.6253,  -2.7778,   1.3528,
          -0.0568,   2.2069,  -1.5284],
        [ -0.3661,  -3.4115,  -4.7638,   4.8061,  -4.4271, -11.1720,   2.8493,
           5.4186,   0.5824,  -2.8319],
        [ -4.3047,  -2.2134,  -0.7932,   2.1109,

In [37]:
1 / 0

ZeroDivisionError: division by zero

---

In [14]:
# OK so it seems to works. However needs to clarify this section:
# feats_s2 = self.encoder2_1(points_list[1], points_list[0], feats_s1, subsampling_list[0])
# feats_s2 = self.encoder2_2(points_list[1], points_list[1], feats_s2, neighbors_list[1])
# feats_s2 = self.encoder2_3(points_list[1], points_list[1], feats_s2, neighbors_list[1])

In [28]:
import sys
sys.path.append("/home/arthur/Documents/Code/Github/Easy-KPConv/examples/scene_segmentation")

from config import make_cfg
from model import create_model


cfg = make_cfg()
easy_model = create_model(cfg).cuda()
print(easy_model.state_dict().keys())
# print(easy_model)

odict_keys(['encoder1_1.conv.weights', 'encoder1_1.conv.bias', 'encoder1_1.conv.kernel_points', 'encoder1_1.norm.norm.weight', 'encoder1_1.norm.norm.bias', 'encoder1_2.unary1.mlp.weight', 'encoder1_2.unary1.mlp.bias', 'encoder1_2.unary1.norm.norm.weight', 'encoder1_2.unary1.norm.norm.bias', 'encoder1_2.conv.conv.weights', 'encoder1_2.conv.conv.bias', 'encoder1_2.conv.conv.kernel_points', 'encoder1_2.conv.norm.norm.weight', 'encoder1_2.conv.norm.norm.bias', 'encoder1_2.unary2.mlp.weight', 'encoder1_2.unary2.mlp.bias', 'encoder1_2.unary2.norm.norm.weight', 'encoder1_2.unary2.norm.norm.bias', 'encoder1_2.unary_shortcut.mlp.weight', 'encoder1_2.unary_shortcut.mlp.bias', 'encoder1_2.unary_shortcut.norm.norm.weight', 'encoder1_2.unary_shortcut.norm.norm.bias', 'encoder2_1.unary1.mlp.weight', 'encoder2_1.unary1.mlp.bias', 'encoder2_1.unary1.norm.norm.weight', 'encoder2_1.unary1.norm.norm.bias', 'encoder2_1.conv.conv.weights', 'encoder2_1.conv.conv.bias', 'encoder2_1.conv.conv.kernel_points', 

In [29]:
from dataset import test_data_loader, train_valid_data_loader


train_loader, val_loader = train_valid_data_loader(cfg, "Area_1")
data_dict = next(iter(train_loader))
print(data_dict.keys())

Testing data is cached in '/home/arthur/Documents/Code/Github/Easy-KPConv/examples/scene_segmentation/dataset_s3dis/s3dis_voxelized_testing_cached_2.5b_1s_2048t'.


dict_keys(['points', 'lengths', 'feats', 'labels', 'batch_size'])


In [59]:
from easy_kpconv.ops.graph_pyramid import build_grid_and_radius_graph_pyramid


feats = data_dict["feats"].cuda()
points = data_dict["points"].cuda()
lengths = data_dict["lengths"].cuda()

graph_pyramid = build_grid_and_radius_graph_pyramid(
    points, 
    lengths, 
    num_stages=easy_model.num_stages, 
    voxel_size=easy_model.voxel_size, 
    search_radius=easy_model.first_radius, 
    neighbor_limits=easy_model.neighbor_limits
)

voxel_size = 0.08
voxel_size = 0.16
voxel_size = 0.32
voxel_size = 0.64
sub_points.shape = torch.Size([60047, 3])
cur_points.shape = torch.Size([195891, 3])
sub_lengths.shape = torch.Size([16])
cur_lengths.shape = torch.Size([16])
search_radius = 0.1
neighbor_limits[i] = 24
sub_points = tensor([[-1.3618, -0.4188,  5.3394],
        [-1.3811, -0.3518,  1.1519],
        [-1.3632, -0.3425,  1.4245],
        ...,
        [ 1.3810, -0.5392,  0.0265],
        [ 1.3660, -0.5225,  1.9777],
        [ 1.3651, -0.5106,  2.4883]], device='cuda:0')
cur_points = tensor([[ 0.5634,  1.0166,  0.2776],
        [ 0.5937,  0.9984,  0.3282],
        [ 0.5792,  1.0172,  0.3383],
        ...,
        [-0.0670,  1.2138,  2.4826],
        [-0.0666,  1.2219,  2.5205],
        [-0.0811,  1.2607,  2.5092]], device='cuda:0')
sub_points.shape = torch.Size([16695, 3])
cur_points.shape = torch.Size([60047, 3])
sub_lengths.shape = torch.Size([16])
cur_lengths.shape = torch.Size([16])
search_radius = 0.2
neighbor_limits

In [60]:
feats = data_dict["feats"].cuda()
points = data_dict["points"].cuda()
lengths = data_dict["lengths"].cuda()

graph_pyramid = build_grid_and_radius_graph_pyramid(
    points, 
    lengths, 
    num_stages=2, 
    voxel_size=easy_model.voxel_size, 
    search_radius=0.5, 
    neighbor_limits=[5, 5]
)

points_list = graph_pyramid["points"]
neighbors_list = graph_pyramid["neighbors"]
subsampling_list = graph_pyramid["subsampling"]
upsampling_list = graph_pyramid["upsampling"]
lengths = graph_pyramid["lengths"]

voxel_size = 0.08
sub_points.shape = torch.Size([60047, 3])
cur_points.shape = torch.Size([195891, 3])
sub_lengths.shape = torch.Size([16])
cur_lengths.shape = torch.Size([16])
search_radius = 0.5
neighbor_limits[i] = 5
sub_points = tensor([[-1.3618, -0.4188,  5.3394],
        [-1.3811, -0.3518,  1.1519],
        [-1.3632, -0.3425,  1.4245],
        ...,
        [ 1.3810, -0.5392,  0.0265],
        [ 1.3660, -0.5225,  1.9777],
        [ 1.3651, -0.5106,  2.4883]], device='cuda:0')
cur_points = tensor([[ 0.5634,  1.0166,  0.2776],
        [ 0.5937,  0.9984,  0.3282],
        [ 0.5792,  1.0172,  0.3383],
        ...,
        [-0.0670,  1.2138,  2.4826],
        [-0.0666,  1.2219,  2.5205],
        [-0.0811,  1.2607,  2.5092]], device='cuda:0')


In [61]:
points_list = graph_pyramid["points"]
neighbors_list = graph_pyramid["neighbors"]
subsampling_list = graph_pyramid["subsampling"]
upsampling_list = graph_pyramid["upsampling"]
lengths = graph_pyramid["lengths"]

feats_s1 = torch.cat([torch.ones_like(feats[:, :1]), feats], dim=1)
feats_s1 = easy_model.encoder1_1(points_list[0], points_list[0], feats_s1, neighbors_list[0])
feats_s1 = easy_model.encoder1_2(points_list[0], points_list[0], feats_s1, neighbors_list[0])

feats_s2 = easy_model.encoder2_1(points_list[1], points_list[0], feats_s1, subsampling_list[0])
feats_s2 = easy_model.encoder2_2(points_list[1], points_list[1], feats_s2, neighbors_list[1])
feats_s2 = easy_model.encoder2_3(points_list[1], points_list[1], feats_s2, neighbors_list[1])

In [88]:
feats_s1.shape

torch.Size([193993, 128])

In [14]:
points_list[1].shape, points_list[0].shape, feats_s1.shape, subsampling_list[0].shape

(torch.Size([53381, 3]),
 torch.Size([180569, 3]),
 torch.Size([180569, 128]),
 torch.Size([53381, 5]))

In [22]:
points_list[1].shape, points_list[1].shape, feats_s2.shape, neighbors_list[1].shape

(torch.Size([53381, 3]),
 torch.Size([53381, 3]),
 torch.Size([53381, 256]),
 torch.Size([53381, 5]))

In [23]:
# cluster = voxel_grid(coords, size=self.grid_size, batch=batch)
# cluster, perm = consecutive_cluster(cluster, return_permutation=True)
# coords = scatter(coords, cluster, dim=0, reduce="mean")
# features = scatter(features, cluster, dim=0, reduce=self.reduce)
# batch = batch[perm]

In [74]:
from easy_kpconv.ops.radius_search import radius_search_pack_mode


q_points = torch.arange(50 * 3, dtype=torch.float, device="cuda").reshape(50, 3)
s_points = torch.arange(50 * 3, dtype=torch.float, device="cuda").reshape(50, 3) + 2
q_lengths = torch.tensor([20, 30], device="cuda")
s_lengths = torch.tensor([20, 30], device="cuda")

radius_search_pack_mode(
    q_points=q_points,
    s_points=s_points,
    q_lengths=q_lengths,
    s_lengths=s_lengths,
    radius=50,
    neighbor_limit=5,
)

# radius_search_pack_mode(
#     q_points=points_list[0],
#     s_points=points_list[1],
#     q_lengths=lengths[0],
#     s_lengths=lengths[1],
#     radius=0.5,
#     neighbor_limit=5,
# )

tensor([[ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 1,  2,  0,  3,  4],
        [ 2,  3,  1,  4,  0],
        [ 3,  4,  2,  5,  1],
        [ 4,  5,  3,  6,  2],
        [ 5,  6,  4,  7,  3],
        [ 6,  7,  5,  8,  4],
        [ 7,  8,  6,  9,  5],
        [ 8,  9,  7, 10,  6],
        [ 9, 10,  8, 11,  7],
        [10, 11,  9, 12,  8],
        [11, 12, 10, 13,  9],
        [12, 13, 11, 14, 10],
        [13, 14, 12, 15, 11],
        [14, 15, 13, 16, 12],
        [15, 16, 14, 17, 13],
        [16, 17, 15, 18, 14],
        [17, 18, 16, 19, 15],
        [18, 19, 17, 16, 15],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [21, 22, 20, 23, 24],
        [22, 23, 21, 24, 20],
        [23, 24, 22, 25, 21],
        [24, 25, 23, 26, 22],
        [25, 26, 24, 27, 23],
        [26, 27, 25, 28, 24],
        [27, 28, 26, 29, 25],
        [28, 29, 27, 30, 26],
        [29, 30, 28, 31, 27],
        [30, 31, 29, 32, 28],
        [31, 32, 30, 33, 29],
        [3

In [71]:
from torch_geometric.nn.pool import radius
from torch_pointcloud.utils.conversion import bincount_to_batch

edge_index = radius(
    s_points, 
    q_points, 
    r=50, 
    batch_x=bincount_to_batch(s_lengths), 
    batch_y=bincount_to_batch(q_lengths),
    max_num_neighbors=5,
)
edge_index[1].reshape(-1, 5)

tensor([[ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 0,  1,  2,  3,  4],
        [ 1,  2,  3,  4,  5],
        [ 2,  3,  4,  5,  6],
        [ 3,  4,  5,  6,  7],
        [ 4,  5,  6,  7,  8],
        [ 5,  6,  7,  8,  9],
        [ 6,  7,  8,  9, 10],
        [ 7,  8,  9, 10, 11],
        [ 8,  9, 10, 11, 12],
        [ 9, 10, 11, 12, 13],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [20, 21, 22, 23, 24],
        [21, 22, 23, 24, 25],
        [22, 23, 24, 25, 26],
        [2

In [59]:
from torch_geometric.nn.pool import radius_graph, radius
from torch_pointcloud.utils.conversion import bincount_to_batch

# feats_s2 = easy_model.encoder2_1(points_list[1], points_list[0], feats_s1, subsampling_list[0])
edge_index = radius(
    points_list[0], 
    points_list[1], 
    r=0.5, 
    batch_x=bincount_to_batch(lengths[0]), 
    batch_y=bincount_to_batch(lengths[1]),
    max_num_neighbors=5,
)

neighbors = edge_index[1].reshape(-1, 5)
neighbors.shape

torch.Size([53381, 5])

In [40]:
subsampling_list[0]

tensor([[  9663,   9649,   9662,   8813,   9664],
        [  9665,   8814,   8815,   9664,   9666],
        [  9676,   9669,   9675,   9677,   8828],
        ...,
        [180375, 180377, 180177, 180179, 180175],
        [180378, 180376, 180180, 180178, 180380],
        [180379, 180381, 180181, 180377, 180383]], device='cuda:0')

In [51]:
points_list[1]

tensor([[-0.7441, -0.4803,  2.6945],
        [-0.7298, -0.4871,  2.7661],
        [-0.7308, -0.4998,  2.9456],
        ...,
        [ 1.3761, -0.6802,  5.6432],
        [ 1.3714, -0.6305,  0.0293],
        [ 1.3650, -0.6045,  5.6206]], device='cuda:0')

In [ ]:
# sub_points = tensor([[-0.7441, -0.4803,  2.6945],
#         [-0.7298, -0.4871,  2.7661],
#         [-0.7308, -0.4998,  2.9456],
#         ...,
#         [ 1.3761, -0.6802,  5.6432],
#         [ 1.3714, -0.6305,  0.0293],
#         [ 1.3650, -0.6045,  5.6206]], device='cuda:0')
# cur_points = tensor([[ 1.2778,  1.0137,  0.0252],
#         [ 1.2681,  1.0236,  0.0074],
#         [ 1.2634,  1.0254,  3.2777],
#         ...,
#         [ 1.2083, -0.0369,  5.2428],
#         [ 1.2108, -0.0379,  5.2932],
#         [ 1.2045, -0.0393,  5.3371]], device='cuda:0')

In [ ]:
# features_1 = self.encoder1_1(x, pos, edge_index_1)
# features_1 = self.encoder1_2(features_1, pos, edge_index_1)

# # Grid pooling to get level 2 points
# pos_2, batch_2, cluster_2 = self.grid_pool_1(pos, batch)

# # For the strided block, we need to handle differently
# # This is the equivalent of:
# # feats_s2 = self.encoder2_1(points_list[1], points_list[0], feats_s1, subsampling_list[0])

# # Find edges between levels 1 and 2
# edge_index_1_to_2 = radius_graph(pos, pos_2, r=self.radius_1, batch_x=batch, batch_y=batch_2)

# # Apply the strided convolution
# features_2 = self.encoder2_1(features_1, pos, pos_2, edge_index_1_to_2)

# # Continue with regular convolutions at level 2
# edge_index_2 = radius_graph(pos_2, r=self.radius_2, batch=batch_2)
# features_2 = self.encoder2_2(features_2, pos_2, edge_index_2)
# features_2 = self.encoder2_3(features_2, pos_2, edge_index_2)
